In [ ]:
#CELL 1 — Import

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    ConfusionMatrixDisplay,
    precision_score, recall_score,
    f1_score, accuracy_score
)

from sklearn.feature_selection import mutual_info_classif

In [ ]:
#CELL 2 — Load, Cleaning & Preprocessing

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# convert numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# missing value
df.dropna(inplace=True)

# drop id column
df.drop(columns=['customerID'], inplace=True)

# encoding target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Outlier Handling (IQR - clipping agar aman secara data mining)
for col in ['tenure', 'MonthlyCharges']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)

print(df.shape)

In [ ]:
#CELL 3 - EDA

fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[0])
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
#CELL 4 - Encoding & Split (Persiapan Data)

categorical_cols = df.select_dtypes(include='object').columns.tolist()

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#CELL 4.1 - FEATURE SELECTION (INFORMATION GAIN)
mi_scores = mutual_info_classif(
    X_train,
    y_train,
    random_state=42
)

mi_result = pd.Series(
    mi_scores,
    index=X.columns
)

mi_result.sort_values(
    ascending=False
).head(10).plot(
    kind='barh',
    figsize=(8,5)
)

plt.title("Information Gain Feature Importance")
plt.xlabel("Score")
plt.show()

# PILIH 15 FITUR TERBAIK

top_features = mi_result.sort_values(
    ascending=False
).head(15).index

print("Top Features:")
print(top_features)

# Hanya ambil fitur terbaik
X_train = X_train[top_features]
X_test = X_test[top_features]

In [ ]:
#4.2 CHI-SQUARE FEATURE SELECTION
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

# Scaling khusus Chi-Square
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Hitung skor Chi-Square
chi_scores, p_values = chi2(X_train_scaled, y_train)

# Convert ke Series
chi_result = pd.Series(
    chi_scores,
    index=X_train.columns
)

# Visualisasi
plt.figure(figsize=(8,5))

chi_result.sort_values(ascending=False).head(10).plot(
    kind='barh',
    color='orange'
)

plt.title("Chi-Square Feature Importance")
plt.xlabel("Chi-Square Score")

plt.grid(axis='x', alpha=0.3)

plt.show()

In [ ]:
#4.3 Gain Ration
def gain_ratio(info_gain, split_info):
    if split_info == 0:
        return 0
    return info_gain / split_info

gain_ratio_scores = {}

for col in X_train.columns:
    
    probs = X_train[col].value_counts(normalize=True)

    split_info = -np.sum(probs * np.log2(probs + 1e-9))

    info_gain = mi_result[col]

    gain_ratio_scores[col] = gain_ratio(info_gain, split_info)

gain_ratio_series = pd.Series(gain_ratio_scores)

gain_ratio_series.sort_values(ascending=False).head(10).plot(
    kind='barh',
    figsize=(8,5),
    color='green'
)

plt.title("Gain Ratio Approximation")
plt.xlabel("Gain Ratio")
plt.show()

In [ ]:
# 4.4 Decision Tree
# DECISION TREE TANPA PRUNING

dt_full = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42
)

dt_full.fit(X_train, y_train)

# DECISION TREE PRE PRUNING

dt_pre = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

dt_pre.fit(X_train, y_train)

# DECISION TREE POST PRUNING
path = dt_full.cost_complexity_pruning_path(
    X_train,
    y_train
)

ccp_alphas = path.ccp_alphas

best_alpha = 0
best_score = 0

# Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for alpha in ccp_alphas:

    clf = DecisionTreeClassifier(
        criterion='entropy',
        ccp_alpha=alpha,
        random_state=42
    )

    score = cross_val_score(
        clf,
        X_train,
        y_train,
        cv=cv,
        scoring='f1'
    ).mean()

    if score > best_score:
        best_score = score
        best_alpha = alpha

print("Best Alpha :", best_alpha)
print("Best CV F1 :", round(best_score, 4))

# Model final post pruning
dt_post = DecisionTreeClassifier(
    criterion='entropy',
    ccp_alpha=best_alpha,
    random_state=42
)

dt_post.fit(X_train, y_train)

results = []

models = {
    "Tanpa Pruning": dt_full,
    "Pre Pruning": dt_pre,
    "Post Pruning": dt_post
}

for name, model in models.items():

    # Prediksi train
    train_pred = model.predict(X_train)

    # Prediksi test
    test_pred = model.predict(X_test)

    # Probabilitas untuk ROC AUC
    test_prob = model.predict_proba(X_test)[:, 1]

    train_acc = accuracy_score(
        y_train,
        train_pred
    )

    test_acc = accuracy_score(
        y_test,
        test_pred
    )

    gap = train_acc - test_acc

    results.append([
        name,
        round(train_acc, 4),
        round(test_acc, 4),
        round(gap, 4),
        round(precision_score(y_test, test_pred), 4),
        round(recall_score(y_test, test_pred), 4),
        round(f1_score(y_test, test_pred), 4),
        round(roc_auc_score(y_test, test_prob), 4)
    ])

comparison = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Train Acc",
        "Test Acc",
        "Gap",
        "Precision",
        "Recall",
        "F1",
        "ROC AUC"
    ]
)

comparison = comparison.sort_values(
    by="F1",
    ascending=False
)

print(comparison)

best_model_name = comparison.iloc[0]["Model"]

print(
    f"\nModel Decision Tree Terbaik: {best_model_name}"
)

In [ ]:
# CELL 4.5 — Decision Tree Evaluation & Visualization
# Visualisasi Perbandingan Performa
metrics_plot = comparison[
    ["Model", "Test Acc", "F1", "ROC AUC"]
]

metrics_plot.set_index("Model").plot(
    kind="bar",
    figsize=(10,6)
)

plt.title("Perbandingan Performa Decision Tree")
plt.ylabel("Score")
plt.ylim(0, 1)

plt.grid(axis="y", alpha=0.3)

plt.legend()

plt.show()

# CELL 4.6 — Visualisasi Gap Overfitting
plt.figure(figsize=(8,5))

plt.bar(
    comparison["Model"],
    comparison["Gap"]
)

plt.title("Perbandingan Gap Overfitting")
plt.ylabel("Train Accuracy - Test Accuracy")

plt.grid(axis='y', alpha=0.3)

plt.show()

# CELL 4.7 — Visualisasi Decision Tree Tanpa Pruning
plt.figure(figsize=(20,10))

plot_tree(
    dt_full,
    feature_names=X_train.columns,
    class_names=['No Churn', 'Churn'],
    filled=True,
    max_depth=3,
    fontsize=8
)

plt.title("Decision Tree Tanpa Pruning")

plt.show()

# CELL 4.8 — Visualisasi Decision Tree Post Pruning
plt.figure(figsize=(20,10))

plot_tree(
    dt_post,
    feature_names=X_train.columns,
    class_names=['No Churn', 'Churn'],
    filled=True,
    max_depth=3,
    fontsize=8
)

plt.title("Decision Tree Post Pruning")

plt.show()

In [ ]:
# 4.9 Classification Report, Confusion Matrix, Node Count
print(
    classification_report(
        y_test,
        best_dt.predict(X_test)
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_dt.predict(X_test),
    display_labels=['No Churn', 'Churn'],
    cmap='Blues'
)

plt.title(
    f'Confusion Matrix - {best_model_name}'
)

plt.show()

print(
    "\nJumlah Node Tanpa Pruning:",
    dt_full.tree_.node_count
)

print(
    "Jumlah Node Post Pruning:",
    dt_post.tree_.node_count
)

In [ ]:
# CELL 5 Modeling Random Forest
# MODELING RANDOM FOREST

rf_pipe = Pipeline([
    ('rf', RandomForestClassifier(
        random_state=42,
        class_weight='balanced',
        criterion='entropy',
        bootstrap=True,
        oob_score=True
    ))
])

param_grid = {
    'rf__n_estimators': [100, 200],

    'rf__max_depth': [5, 8, 10],

    'rf__min_samples_split': [10, 20],

    'rf__min_samples_leaf': [5, 10],

    'rf__max_features': ['sqrt']
}

grid_rf = GridSearchCV(
    rf_pipe,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

print("Best Parameter:")
print(grid_rf.best_params_)

# OOB SCORE

rf_model = best_rf.named_steps['rf']

print("\nOOB Score:", rf_model.oob_score_)

# VISUALISASI SATU POHON

tree = rf_model.estimators_[0]

plt.figure(figsize=(20,10))

plot_tree(
    tree,
    feature_names=X.columns,
    class_names=['No Churn', 'Churn'],
    max_depth=3,
    filled=True
)

plt.title("Random Forest Tree Visualization")

plt.show()